In [1]:
import pandas as pd
import numpy as np

# 1. 캐글 비행기 데이터 불러오기
print("전 세계 비행기 데이터를 불러옵니다...")
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/opensky_states_snapshot.csv')

# 2. 하늘을 날고 있는 비행기만 추려내기 (땅에 있는 건 제외)
# vertical_rate(수직 비율) 데이터가 비어있는(결측치) 비행기도 제외합니다.
df_flying = df[df['on_ground'] == False].dropna(subset=['vertical_rate'])
print(f"현재 하늘을 날고 있는 비행기: {len(df_flying)}대")

# 3. 수학의 힘! 평균(μ)과 표준편차(σ) 구하기
mean_vr = np.mean(df_flying['vertical_rate'])
std_vr = np.std(df_flying['vertical_rate'])

print(f"정상 비행기의 평균 승강률: {mean_vr:.2f} m/s")
print(f"승강률의 표준편차: {std_vr:.2f}")

# 4. 표준화 공식(Z-score) 적용하기
# (현재 속도 - 평균) / 표준편차
df_flying['z_score'] = (df_flying['vertical_rate'] - mean_vr) / std_vr

# 5. 하인리히의 '징조(300)' 찾기!
# Z-score가 -3 이하이면, 확률적으로 0.13%밖에 안 되는 극단적인 급강하 상태입니다.
danger_planes = df_flying[df_flying['z_score'] <= -3]

# 6. 결과 발표
print(f"삐용삐용! 이상 징후(급강하)를 보이는 비행기 {len(danger_planes)}대 발견!")
print("위험 비행기 명단 (항공편명, 국적, 현재 하강속도, 위험지수)")
danger_planes[['callsign', 'origin_country', 'vertical_rate', 'z_score']]


전 세계 비행기 데이터를 불러옵니다...
현재 하늘을 날고 있는 비행기: 6832대
정상 비행기의 평균 승강률: -0.01 m/s
승강률의 표준편차: 5.12
삐용삐용! 이상 징후(급강하)를 보이는 비행기 33대 발견!
위험 비행기 명단 (항공편명, 국적, 현재 하강속도, 위험지수)


,callsign,origin_country,vertical_rate,z_score
122,DMQIS,Germany,-20.81,-4.065893
536,BBC372,Bangladesh,-15.93,-3.111878
660,BKP151,Thailand,-16.58,-3.238950
805,EJU31YN,Austria,-21.78,-4.255523
1027,MAC342,Morocco,-16.91,-3.303463
1029,CFE23DB,United Kingdom,-15.93,-3.111878
1256,S5DOT,Slovenia,-18.86,-3.684678
1604,FDB549,United Arab Emirates,-17.56,-3.430535
1665,VLG3YM,Spain,-24.06,-4.701251
1858,EWG19K,Germany,-16.26,-3.176392


In [7]:
import requests
import pandas as pd

def fetch_korea_flights():
    # OpenSky Network API 엔드포인트
    url = "https://opensky-network.org/api/states/all"

    # 한반도 상공 Bounding Box (위도 33~39, 경도 124~132)
    params = {
        "lamin": 33.0,
        "lamax": 39.0,
        "lomin": 124.0,
        "lomax": 132.0
    }

    # 사용자 인증 정보 (아이디, 비밀번호)
    # 주의: 실제 운영 환경에서는 환경 변수(os.environ)를 사용하는 것이 안전합니다.
    auth = ("API키", "비밀번호")

    try:
        # API 요청 보내기
        response = requests.get(url, params=params, auth=auth)
        response.raise_for_status() # HTTP 오류 발생 시 예외 처리

        # JSON 데이터 파싱
        data = response.json()

        # 데이터가 존재하는지 확인 ('states' 키 안에 비행기 정보가 리스트 형태로 들어있음)
        if data['states'] is not None:
            # 요청하신 컬럼명 리스트
            columns = [
                'icao24', 'callsign', 'origin_country', 'time_position',
                'last_contact', 'longitude', 'latitude', 'baro_altitude',
                'on_ground', 'velocity', 'true_track', 'vertical_rate',
                'sensors', 'geo_altitude', 'squawk', 'spi', 'position_source'
            ]

            # Pandas 데이터프레임으로 변환
            df = pd.DataFrame(data['states'], columns=columns)

            # 콜사인(callsign) 문자열 끝에 붙은 공백 제거 (데이터 전처리)
            df['callsign'] = df['callsign'].str.strip()

            return df
        else:
            print("현재 지정된 한반도 상공에 조회되는 비행기가 없습니다.")
            return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print(f"API 요청 중 에러가 발생했습니다: {e}")
        return None

# 함수 실행 및 결과 확인
if __name__ == "__main__":
    korea_flights_df = fetch_korea_flights()

    if korea_flights_df is not None and not korea_flights_df.empty:
        print(f"✅ 총 {len(korea_flights_df)}대의 비행기 데이터를 성공적으로 가져왔습니다.\n")
        # 데이터프레임의 상위 5개 행 출력
        display(korea_flights_df.head()) # 주피터 노트북 환경이 아니라면 print(korea_flights_df.head()) 사용

KeyboardInterrupt: 

In [ ]:
import folium
import pandas as pd

def visualize_flights_on_map(df):
    # 데이터가 비어있는지 확인
    if df is None or df.empty:
        print("시각화할 비행기 데이터가 없습니다.")
        return None

    # 한반도 중심부 좌표 설정 (충청북도 인근)
    center_lat, center_lon = 36.0, 128.0

    # 1. 초기 지도 생성 (zoom_start로 초기 확대 정도 설정)
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7, tiles='CartoDB positron')

    # 2. 데이터프레임의 각 행(비행기 1대)을 반복하며 지도에 마커 추가
    for index, row in df.iterrows():
        lat = row['latitude']
        lon = row['longitude']

        # 위치 정보가 없는 데이터는 건너뜀 (결측치 처리)
        if pd.isna(lat) or pd.isna(lon):
            continue

        # 툴팁 및 팝업에 표시할 정보 추출 및 정리
        callsign = row['callsign'] if row['callsign'] else "Unknown"
        altitude = row['baro_altitude'] if pd.notna(row['baro_altitude']) else "N/A"
        velocity = row['velocity'] if pd.notna(row['velocity']) else "N/A"
        heading = row['true_track'] # 비행기 진행 방향 (각도)

        # 팝업을 클릭했을 때 나올 HTML 정보창
        popup_html = f"""
        <div style="width: 150px;">
            <b>콜사인:</b> {callsign}<br>
            <b>고도:</b> {altitude} m<br>
            <b>속도:</b> {velocity} m/s
        </div>
        """

        # 3. 비행기 아이콘 마커 생성
        # FontAwesome(fa)의 'plane' 아이콘을 사용하고, 진행 방향(heading)에 맞춰 아이콘을 회전(angle)시킵니다.
        # 주의: folium 버전이나 브라우저에 따라 angle 속성이 완벽히 적용되지 않을 수 있습니다.
        icon_angle = heading if pd.notna(heading) else 0

        folium.Marker(
            location=[lat, lon],
            tooltip=f"<b>{callsign}</b> (클릭하여 상세정보 보기)", # 마우스를 올렸을 때 나오는 글씨
            popup=folium.Popup(popup_html, max_width=250),
            icon=folium.Icon(icon='plane', prefix='fa', color='blue', angle=icon_angle)
        ).add_to(m)

    return m

# --- 실행 부분 ---
# 주피터 노트북(Jupyter Notebook) 환경이라면 아래 코드 실행 시 지도가 바로 출력됩니다.
flight_map = visualize_flights_on_map(korea_flights_df)

if flight_map:
    display(flight_map)

    # HTML 파일로 저장하여 웹 브라우저에서 열어볼 수도 있습니다.
    flight_map.save("korea_flights_map.html")
    print("\n✅ 'korea_flights_map.html' 파일로 지도가 저장되었습니다. 웹 브라우저에서 열어보세요!")


✅ 'korea_flights_map.html' 파일로 지도가 저장되었습니다. 웹 브라우저에서 열어보세요!


In [ ]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd
import numpy as np

# 1. 페이지 기본 설정
st.set_page_config(page_title="AI 스마트 항공 관제탑", layout="wide")

# 2. 데이터 수집 함수
@st.cache_data(ttl=30)
def fetch_flight_data():
    url = "https://opensky-network.org/api/states/all"
    params = {"lamin": 33.0, "lamax": 39.0, "lomin": 124.0, "lomax": 132.0}
    auth = ("lee lee-api-client", "Abcd1234")

    try:
        response = requests.get(url, params=params, auth=auth, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data['states'] is not None:
            columns = ['icao24', 'callsign', 'origin_country', 'time_position',
                       'last_contact', 'longitude', 'latitude', 'baro_altitude',
                       'on_ground', 'velocity', 'true_track', 'vertical_rate',
                       'sensors', 'geo_altitude', 'squawk', 'spi', 'position_source']
            df = pd.DataFrame(data['states'], columns=columns)
            df['callsign'] = df['callsign'].str.strip()
            return df
        else:
            return pd.DataFrame()
    except Exception as e:
        st.error(f"API 요청 중 에러가 발생했습니다: {e}")
        return None

# ==========================================
# 3. 화면 UI 구성 시작
# ==========================================

col1, col2 = st.columns([8, 2])
with col1:
    st.title("🚨 AI 스마트 항공 관제탑 (한반도 상공)")
with col2:
    st.write("")
    if st.button("🔄 수동 새로고침", use_container_width=True):
        st.cache_data.clear()
        st.rerun()

st.markdown("---")

with st.spinner("한반도 상공의 실시간 레이더망을 스캔하는 중입니다..."):
    df = fetch_flight_data()

if df is not None and not df.empty:

    # [데이터 전처리]
    analysis_df = df.dropna(subset=['latitude', 'longitude', 'vertical_rate']).copy()

    vr_mean = analysis_df['vertical_rate'].mean()
    vr_std = analysis_df['vertical_rate'].std()

    if pd.isna(vr_std) or vr_std == 0:
        analysis_df['z_score'] = 0.0
    else:
        analysis_df['z_score'] = (analysis_df['vertical_rate'] - vr_mean) / vr_std

    # ==========================================
    # 🛠️ [핵심 포인트] 시뮬레이션 제어판 (슬라이더)
    # ==========================================
    st.subheader("🛠️ 시뮬레이션 제어판 (경보 민감도 조절)")

    # st.slider를 통해 학생이 직접 기준값을 -5.0 부터 5.0 까지 조절 가능 (기본값 -3.0)
    threshold = st.slider("⚠️ 위험 경보 Z-score 기준값 조절", min_value=-5.0, max_value=5.0, value=-3.0, step=0.5)

    st.info(f"💡 **현재 설정:** Z-score가 **{threshold}** 이하인 비행기를 '위험(급강하)'으로 간주하여 경보를 울립니다.")

    # 슬라이더에서 받은 threshold 변수를 조건문에 적용!
    analysis_df['status'] = np.where(analysis_df['z_score'] <= threshold, '위험(급강하)', '정상')
    danger_count = (analysis_df['status'] == '위험(급강하)').sum()

    # ==========================================

    # [화면 중앙 - 핵심 경보 기능 (Metric)]
    st.markdown("<br>", unsafe_allow_html=True)
    st.subheader("📊 실시간 항공 상태 요약")
    m1, m2, m3 = st.columns(3)

    with m1:
        st.metric(label="총 탐지된 비행기", value=f"{len(analysis_df)} 대")
    with m2:
        delta_color = "inverse" if danger_count > 0 else "off"
        st.metric(label="🚨 현재 위험(급강하) 비행기", value=f"{danger_count} 대", delta="- 주의 필요" if danger_count > 0 else "안전", delta_color=delta_color)
    with m3:
        st.metric(label="평균 수직 승강률", value=f"{vr_mean:.2f} m/s")

    st.markdown("---")

    # [지도 및 하단 데이터 표 시각화]
    col_map, col_table = st.columns([1, 1]) # 지도를 표 위가 아니라 옆에 배치해서 한눈에 보게 수정

    with col_map:
        st.subheader("📍 레이더망")
        st.map(analysis_df[['latitude', 'longitude']], zoom=5, use_container_width=True)

    with col_table:
        st.subheader("📋 상세 표")
        display_cols = ['callsign', 'status', 'z_score', 'vertical_rate', 'baro_altitude']
        final_df = analysis_df[display_cols].sort_values(by='z_score') # 위험한 비행기가 위로 오게 정렬

        st.dataframe(
            final_df.style.format({"z_score": "{:.2f}", "vertical_rate": "{:.2f}"})
                          .map(lambda x: 'background-color: #ffcccc; color: red; font-weight: bold' if x == '위험(급강하)' else '', subset=['status']),
            use_container_width=True, height=400
        )

else:
    st.warning("📡 현재 감지된 비행기가 없거나 API 호출 대기 시간입니다.")

In [ ]:
print("🌐 관제탑 서버를 인터넷에 연결하는 중입니다... (약 5~10초 소요)")

!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > tunnel.log 2>&1 &



# 4. 링크가 생성될 때까지 잠시 대기 후 출력

time.sleep(7)

print("\n👇 아래에 생성된 링크(https://~~~.trycloudflare.com)를 클릭하세요! 👇")

!grep -o 'https://.*\.trycloudflare.com' tunnel.log




🌐 관제탑 서버를 인터넷에 연결하는 중입니다... (약 5~10초 소요)

👇 아래에 생성된 링크(https://~~~.trycloudflare.com)를 클릭하세요! 👇
https://non-agency-budgets-eos.trycloudflare.com
